In [19]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, recall_score
from sklearn.metrics import make_scorer

In [20]:
# 1. Load the breast cancer dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target



In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [28]:
# 3. Create the Pipeline (Scaling is critical for KNN here!)
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# 4. Set up the hyperparameter grid for KNN
param_grid = {
    'knn__n_neighbors':[3,5,7,9,11,13,15,17,19]
    # 'knn__weights': ['uniform', 'distance']
}

# Create a scorer that targets the 'malignant' class (0)
malignant_recall_scorer = make_scorer(recall_score, pos_label=0)

# 5. Initialize GridSearchCV optimizing for RECALL
grid_search = GridSearchCV(
    knn_pipeline,
    param_grid,
    cv=5,
    # scoring='recall',  # Prioritizes finding all positive cancer cases
    scoring=malignant_recall_scorer  # it prioritizes finding cancer
)


In [29]:
# 6. Fit and predict
grid_search.fit(X_train, y_train)
y_pred = grid_search.predict(X_test)

# 7. Print the final metrics
print("Best Parameters:", grid_search.best_params_)
print(f"Best Training Recall: {grid_search.best_score_:.3f}")
print("\n--- Final Test Evaluation ---")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

Best Parameters: {'knn__n_neighbors': 9}
Best Training Recall: 0.931

--- Final Test Evaluation ---
              precision    recall  f1-score   support

   malignant       1.00      0.91      0.95        53
      benign       0.95      1.00      0.97        90

    accuracy                           0.97       143
   macro avg       0.97      0.95      0.96       143
weighted avg       0.97      0.97      0.96       143

